<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/M08-deep-learning/AT%26T_logo_2016.svg" alt="AT&T LOGO" width="50%" />

# AT&T Spam Detector -- Deep Learning Project

## Company Description

AT&T Inc. is an American multinational telecommunications holding company headquartered in Dallas, Texas. It is the world's largest telecommunications company by revenue.

## Objective

Build a spam detector that can automatically flag spam SMS messages based solely on their content. We will train and compare three specific deep learning models:

1. **Baseline M0** -- Traditional Machine Learning (Logistic Regression)
2. **Model A** -- Simple Neural Network (Embedding + Dense)
3. **Model B** -- Transfer Learning with Sentence-Transformers

We will evaluate and compare them using precision, recall, f1-score, and accuracy, and conclude with qualitative examples of classification.

## 0. Install Dependencies

Run this cell once to ensure all required packages are installed.

In [ ]:
!pip install -q tensorflow tf-keras wordcloud scikit-learn matplotlib seaborn sentence-transformers


## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    LSTM,
    Dropout,
    Bidirectional,
    GlobalAveragePooling1D,
    Input,
)
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow version: {tf.__version__}")


## 2. Data Loading and Exploration

We load `spam.csv`, keep only the relevant columns, and rename them.

The dataset contains 5,572 SMS messages labeled as **ham** (legitimate) or **spam**.

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")

# Keep only relevant columns and rename them
df = df[["v1", "v2"]]
df.columns = ["label", "message"]

print(f"Dataset shape: {df.shape}")
df.head(10)


### 2.1 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["label"].value_counts()
colors = ["#2196F3", "#F44336"]
counts.plot(kind="bar", color=colors, edgecolor="black", ax=ax)
ax.set_title("Class Distribution: Ham vs Spam", fontsize=14, fontweight="bold")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nHam: {counts['ham']} ({counts['ham']/len(df)*100:.1f}%)")
print(f"Spam: {counts['spam']} ({counts['spam']/len(df)*100:.1f}%)")
print(f"Imbalance ratio: {counts['ham']/counts['spam']:.1f}:1")


### 2.2 Message Length Analysis

Compare the distribution of message lengths between ham and spam.

In [ ]:
df["msg_length"] = df["message"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, label in enumerate(["ham", "spam"]):
    subset = df[df["label"] == label]
    axes[i].hist(subset["msg_length"], bins=50, color=colors[i], edgecolor="black", alpha=0.8)
    axes[i].set_title(f"Message Length Distribution -- {label.upper()}", fontweight="bold")
    axes[i].set_xlabel("Character count")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(subset["msg_length"].mean(), color="black", linestyle="--",
                    label=f"Mean: {subset['msg_length'].mean():.0f}")
    axes[i].legend()

plt.tight_layout()
plt.show()

print(df.groupby("label")["msg_length"].describe().round(1))


### 2.3 Word Clouds

Visualize the most frequent words in spam vs. ham messages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, label in enumerate(["ham", "spam"]):
    text = " ".join(df[df["label"] == label]["message"].values)
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap="Blues" if label == "ham" else "Reds",
        max_words=100,
    ).generate(text)
    axes[i].imshow(wc, interpolation="bilinear")
    axes[i].set_title(f"Word Cloud -- {label.upper()}", fontsize=14, fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## 3. Text Preprocessing

Steps:
- **Clean text**: regex `str.replace(r"[\W_]+", " ")` removes punctuation but explicitly keeps alphanumeric characters (numbers are very important in SMS spam! e.g. shortcodes, phone numbers, prices).
- Encode labels (`ham=0`, `spam=1`)
- Train/test split (80/20, stratified to preserve the 13% spam distribution)
- **Tokenization**: convert text into integer sequences (each word = an integer).
- **Padding**: `pad_sequences` to make all inputs the same length (a requirement for Neural Networks).

In [ ]:
# Encode labels
df["label_enc"] = df["label"].map({"ham": 0, "spam": 1})

# Clean text: remove special characters, underscores, and lowercase
df["clean_text"] = df["message"].str.replace(r"[\W_]+", " ", regex=True)
df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x.lower())

# Train/test split (stratified to preserve class balance)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label_enc"],
    test_size=0.2,
    random_state=SEED,
    stratify=df["label_enc"],
)

print(f"Training set: {len(X_train_text)} samples")
print(f"Test set:     {len(X_test_text)} samples")
print(f"\nTraining class distribution:\n{y_train.value_counts().to_string()}")

# Tokenization
MAX_WORDS = 10000
MAX_LEN = 150

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print(f"\nVocabulary size: {min(len(tokenizer.word_index)+1, MAX_WORDS)}")
print(f"Padded sequence shape: {X_train_pad.shape}")


## 4. Baseline -- Traditional Machine Learning

Before using Deep Learning, let's establish a baseline using traditional machine learning (Logistic Regression) on TF-IDF vectorized text.

This will help us measure the actual added value of neural networks.

In [ ]:
# TF-IDF Vectorization for classical ML
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=["ham", "spam"]))

results_lr = {"model": "Baseline (LogReg)", "accuracy": accuracy_score(y_test, y_pred_lr), "precision": precision_score(y_test, y_pred_lr), "recall": recall_score(y_test, y_pred_lr), "f1": f1_score(y_test, y_pred_lr), "y_pred": y_pred_lr}


## 5. Model A -- Simple Neural Network

Architecture: `Embedding` -> `GlobalAveragePooling1D` -> `Dense(24, relu)` -> `Dense(1, sigmoid)`

This is a deliberately simple baseline to assess how far we can go without recurrent layers.

In [ ]:
model_a = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=32, input_length=MAX_LEN),
    GlobalAveragePooling1D(),
    Dense(24, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model_a.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model_a.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history_a = model_a.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1,
)


### Training Curves -- Model A

In [ ]:
def plot_training_curves(history, model_name):
    """Plot loss and accuracy curves for a given training history."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(history.history["loss"], label="Train Loss", linewidth=2)
    axes[0].plot(history.history["val_loss"], label="Val Loss", linewidth=2)
    axes[0].set_title(f"{model_name} -- Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy", linewidth=2)
    axes[1].set_title(f"{model_name} -- Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_training_curves(history_a, "Model A (Simple NN)")


### Evaluation -- Model A

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, use_predict=True):
    """Evaluate a model and print the classification report."""
    if use_predict:
        y_pred_prob = model.predict(X_test)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    else:
        y_pred = model  # already predictions

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"=== {model_name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))

    return {"model": model_name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "y_pred": y_pred}

results_a = evaluate_model(model_a, X_test_pad, y_test, "Model A (Simple NN)")


## 6. Model B -- Transfer Learning (Sentence-Transformers)

We use the **all-MiniLM-L6-v2** model from Sentence-Transformers to embed each message into a 384-dimensional vector. A simple dense classifier is trained on top.

This approach leverages a model pre-trained on a very large text corpus, which should provide rich semantic features even with limited data.

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

from sentence_transformers import SentenceTransformer

# Load the sentence-transformers model
print("Loading sentence-transformers model (all-MiniLM-L6-v2)...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode train and test messages into dense vectors (384 dimensions)
print("Encoding training messages...")
X_train_st = st_model.encode(X_train_text.tolist(), show_progress_bar=True, batch_size=64)

print("Encoding test messages...")
X_test_st = st_model.encode(X_test_text.tolist(), show_progress_bar=True, batch_size=64)

print(f"\nEmbedding shape: {X_train_st.shape}")


In [ ]:
# Build a Dense classifier on top of the sentence embeddings
EMBED_DIM = X_train_st.shape[1]  # 384

model_c = Sequential([
    Input(shape=(EMBED_DIM,)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model_c.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model_c.summary()


In [ ]:
early_stop_c = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history_c = model_c.fit(
    X_train_st, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop_c],
    verbose=1,
)


### Training Curves -- Model B

In [ ]:
plot_training_curves(history_c, "Model C (Sentence-Transformers)")


### Evaluation -- Model B

In [ ]:
results_c = evaluate_model(model_c, X_test_st, y_test, "Model C (Sentence-Transformers)")


## 7. Model Comparison

We compare all three models on the test set using accuracy, precision, recall, and F1-score.

In [ ]:
# Build comparison table
comparison = pd.DataFrame([
    {k: v for k, v in results_lr.items() if k != "y_pred"},
    {k: v for k, v in results_a.items() if k != "y_pred"},
    {k: v for k, v in results_c.items() if k != "y_pred"},
])

comparison = comparison.set_index("model")

# Format as percentages
comparison_pct = comparison.apply(lambda x: x.map("{:.2%}".format))

print("=" * 60)
print("MODEL COMPARISON -- TEST SET PERFORMANCE")
print("=" * 60)
print(comparison_pct.to_string())
print()

# Highlight best model
best_model = comparison["f1"].idxmax()
print(f"Best model by F1-score: {best_model} ({comparison.loc[best_model, 'f1']:.2%})")


### Confusion Matrices -- Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

all_results = [results_a, results_c]
names = ["Model A\n(Simple NN)", "Model B\n(Sentence-Transformers)"]

for i, (res, name) in enumerate(zip(all_results, names)):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Ham", "Spam"],
        yticklabels=["Ham", "Spam"],
        ax=axes[i],
    )
    axes[i].set_title(f"{name}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

plt.suptitle("Confusion Matrices -- DL Models", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 8. Conclusion & Qualitative Analysis

We trained and compared three approaches for SMS spam detection:

- **M0 (Logistic Regression)**: excellent baseline, but lower recall.
- **Model A** (Simple NN): fast to train, solid DL entry point.
- **Model B** (Sentence-Transformers Transfer Learning): leverages pre-trained global semantic embeddings for the best possible F1-score and generalization.

For production, **Model B** is recommended.

Here are some concrete prediction examples of the best model vs reality:

In [ ]:
# Map predictions back to labels
best_preds = results_c["y_pred"]
mapping = {0: "ham", 1: "spam"}

results_df = pd.DataFrame({
    "message_text": X_test_text,
    "actual_label": y_test.map(mapping),
    "predicted_label": pd.Series(best_preds, index=y_test.index).map(mapping)
})

results_df["correct"] = results_df["actual_label"] == results_df["predicted_label"]

print("=== 🎯 EXAMPLES OF CORRECT PREDICTIONS ===")
good_preds = results_df[results_df["correct"]].sample(5, random_state=SEED)
for i, row in good_preds.iterrows():
    print(f"\nTEXT: {row['message_text'][:100]}...")
    print(f"ACTUAL: {row['actual_label']}  |  PREDICTED: {row['predicted_label']}")

print("\n" + "="*50 + "\n")

print("=== ❌ EXAMPLES OF ERRORS (FALSE NEGATIVES / FALSE POSITIVES) ===")
errors = results_df[~results_df["correct"]]
if len(errors) > 0:
    error_sample = errors.head(5)
    for i, row in error_sample.iterrows():
        print(f"\nTEXT: {row['message_text'][:100]}...")
        print(f"ACTUAL: {row['actual_label']}  |  PREDICTED: {row['predicted_label']}  (INCORRECT)")
else:
    print("No errors found in this sample!")
